# CKD Pipeline Runner — Revision 1 (MLWA-D-26-01812) — Google Colab

Reproduces the primary analysis, all reviewer-requested Revision 1 analyses, and manuscript figures.

In [ ]:
# Pin the environment used for the manuscript reproduction.
import importlib.metadata as md, os, subprocess, sys
PINNED = {
    'numpy':'2.4.6','pandas':'2.2.3','scipy':'1.16.3','scikit-learn':'1.6.1',
    'xgboost':'3.4.1','lightgbm':'4.7.0','imbalanced-learn':'0.14.2',
    'statsmodels':'0.15.0','joblib':'1.6.0','threadpoolctl':'3.7.0'
}
def ver(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return None
mismatch = {p:ver(p) for p,v in PINNED.items() if ver(p) != v}
if mismatch:
    print('Installing pinned versions:', mismatch)
    subprocess.check_call([sys.executable,'-m','pip','install','-q', *[f'{p}=={PINNED[p]}' for p in mismatch]])
    print('If Colab changed an already-loaded package, restart the runtime and Run all once more.')
for p,v in PINNED.items():
    print(f'{p:<18} {ver(p):<10} ' + ('OK' if ver(p)==v else 'MISMATCH'))


In [ ]:
# Mount Drive and locate/extract the final 42-module project.
import os, sys, time, zipfile
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/CKD-Paper-Final')
ZIP_PATH = BASE / 'ckd-project.zip'
EXPECTED_MODULES = 42
def nmods(p): return len(list((p/'modules').glob('[0-9][0-9]_*.py'))) if (p/'modules').is_dir() else 0
def candidates():
    return [p.parent for p in BASE.glob('**/pipeline_runner.py') if len(p.parent.relative_to(BASE).parts)<=3 and nmods(p.parent)==EXPECTED_MODULES]
hits = candidates()
if not hits:
    if not ZIP_PATH.exists(): raise FileNotFoundError(f'Upload {ZIP_PATH.name} to {BASE}')
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(BASE)
    hits = candidates()
if not hits: raise FileNotFoundError('No complete 42-module CKD project found after extraction.')
PROJECT_ROOT = min(hits, key=lambda p: len(p.relative_to(BASE).parts))
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('PROJECT_ROOT:', PROJECT_ROOT)
print('Modules:', nmods(PROJECT_ROOT))


In [ ]:
# Initialize the modular pipeline.
import importlib, pipeline_runner
importlib.reload(pipeline_runner)
from pipeline_runner import CKDPipelineRunner, banner
runner = CKDPipelineRunner(PROJECT_ROOT)
RUN_DATA_PREPARATION = True
PIPELINE_T0 = time.time()


In [ ]:
# Primary analysis: modules 00–25.
runner.setup()
if RUN_DATA_PREPARATION: runner.prepare_data()
runner.load_development_data()
runner.build_preprocessor()
runner.define_candidate_models()
runner.define_metrics()
runner.run_nested_cv()

# Reproduction checkpoint: Table 2.
published_cv = {
 'Random Forest':(0.7879,0.1297), 'XGBoost':(0.7871,0.1320),
 'HistGradientBoosting':(0.7819,0.1237), 'LightGBM':(0.7817,0.1572),
 'Logistic Regression':(0.7808,0.1364)}
mc = runner.get('model_comparison').set_index('model')
for m,(auc,brier) in published_cv.items():
    got=(round(float(mc.loc[m,'mean_roc_auc']),4), round(float(mc.loc[m,'mean_brier']),4))
    print(m, got, 'MATCH' if got==(auc,brier) else 'DIFFERENT')

runner.run_bootstrap_uncertainty()
runner.run_paired_model_comparison()
runner.tune_final_model()
runner.calibrate_model()
runner.freeze_threshold()
runner.fit_final_model()
runner.build_temporal_cohort()
runner.run_temporal_validation()

# Reproduction checkpoint: frozen primary result.
checks = {
 'model': ('HistGradientBoosting', runner.get('SELECTED_MODEL_NAME')),
 'calibration': ('Sigmoid', runner.get('BEST_CALIBRATION')),
 'threshold': (0.1239, round(float(runner.get('SCREENING_THRESHOLD')),4)),
 'temporal_auc': (0.7949, round(float(runner.get('temporal_unweighted')['roc_auc']),4)),
 'temporal_sensitivity': (0.8251, round(float(runner.get('temporal_unweighted')['sensitivity']),4)),
 'temporal_specificity': (0.6037, round(float(runner.get('temporal_unweighted')['specificity']),4)),
 'weighted_sensitivity': (0.7671, round(float(runner.get('temporal_weighted')['sensitivity']),4)),
}
for k,(expected,got) in checks.items(): print(k, got, 'MATCH' if got==expected else 'DIFFERENT')

runner.run_temporal_calibration()
runner.run_decision_curve()
runner.run_age_sensitivity()
runner.run_outcome_component_sensitivity()
runner.run_feature_set_sensitivity()
runner.run_subgroup_analysis()
runner.run_stratified_thresholds()
runner.run_imbalance_sensitivity()
runner.run_survey_weighted_evaluation()
runner.run_permutation_importance()
runner.finalize_outputs()


In [ ]:
# Reviewer-requested Revision 1 analyses: modules 26–39.
runner.revision_setup()
runner.run_cohort_description()
runner.run_model_selection_revision()
runner.run_comparators_and_ablation()
runner.run_survey_weighted_threshold()
runner.run_design_based_uncertainty()
runner.run_calibration_uncertainty()
runner.run_subgroup_uncertainty_and_policies()
runner.run_age_dependence()
runner.run_alcohol_missingness()
runner.run_screening_scenarios()
runner.run_correlated_importance()
runner.run_blood_pressure_protocol()
runner.finalize_revision_outputs()


In [ ]:
# Export figure data and generate Figures 1–6 plus Supplementary Figure S1.
runner.export_figure_data()
runner.make_figures()
print('Primary results :', runner.get('RESULTS_DIR'))
print('Revision results:', runner.get('REVISION_DIR'))
print(f'Total runtime: {(time.time()-PIPELINE_T0)/60:.1f} minutes')
